In [18]:
import pandas as pd
import numpy as np

DATA_FOLDER = "./"
df = pd.read_pickle(DATA_FOLDER + "df_fe_for_ensamble_best_customers_0c.pickle")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()
#df = df[df["product_id"].isin(product_ids)]
df = df.sort_values(by=["date_id", "product_id"])
df["customer_id"] = 0
df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)


In [19]:
df.describe()

,product_id,cust_request_qty,cust_request_tn,tn,stock_final,sku_size,year,mes,quarter,date_id,...,prod_tn_lag_1_x_tn_lag_11,prod_tn_lag_1_x_tn_lag_8,prod_tn_lag_3_x_tn_lag_2,prod_tn_lag_3_x_tn_lag_11,prod_tn_lag_3_x_tn_lag_8,prod_tn_lag_2_x_tn_lag_11,prod_tn_lag_2_x_tn_lag_8,prod_tn_lag_11_x_tn_lag_8,customer_id,target
count,31522.000000,31522.000000,31522.000000,31522.000000,13691.000000,31229.000000,31522.000000,31522.000000,31522.000000,31522.000000,...,1.920900e+04,2.224000e+04,2.787000e+04,1.920900e+04,2.224000e+04,1.920900e+04,2.224000e+04,1.920900e+04,31522.0,29076.000000
mean,20535.827073,200.806865,42.924751,42.033772,19.478148,476.827881,2018.037688,6.575471,2.524840,18.027727,...,1.405697e+04,1.394960e+04,1.369290e+04,1.435157e+04,1.411351e+04,1.426182e+04,1.402111e+04,1.518359e+04,0.0,42.737881
std,347.109552,124.339898,113.127739,109.374512,55.627438,883.449097,0.816015,3.452354,1.118599,10.355680,...,1.006608e+05,1.006127e+05,1.004995e+05,9.949145e+04,9.961061e+04,1.020440e+05,9.971995e+04,1.012742e+05,0.0,111.549156
min,20001.000000,0.000000,0.000000,0.000000,-27.311359,1.000000,2017.000000,1.000000,1.000000,0.000000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000
25%,20239.000000,106.000000,2.221153,2.211540,1.160960,90.000000,2017.000000,4.000000,2.000000,9.000000,...,6.883162e+00,6.546921e+00,6.283891e+00,7.520869e+00,7.176875e+00,7.182292e+00,6.957675e+00,8.685013e+00,0.0,2.222740
50%,20495.000000,185.000000,9.652330,9.598450,5.419600,250.000000,2018.000000,7.000000,3.000000,18.000000,...,1.160452e+02,1.088569e+02,1.063643e+02,1.199704e+02,1.132123e+02,1.201063e+02,1.119023e+02,1.358740e+02,0.0,9.635365
75%,20812.000000,281.000000,29.836285,29.572310,17.584541,475.000000,2019.000000,10.000000,4.000000,27.000000,...,1.034533e+03,9.596051e+02,9.119448e+02,1.041381e+03,9.797971e+02,1.059134e+03,9.570754e+02,1.129115e+03,0.0,29.966728
max,21299.000000,756.000000,2423.708740,2295.198242,1562.024536,10000.000000,2019.000000,12.000000,4.000000,35.000000,...,3.009615e+06,4.261805e+06,4.161229e+06,3.366470e+06,3.375448e+06,3.853622e+06,3.781657e+06,3.374882e+06,0.0,2295.198242


In [20]:
df[["fecha", "date_id"]]

,fecha,date_id
0,2017-01,0
1,2017-01,0
2,2017-01,0
3,2017-01,0
4,2017-01,0
...,...,...
31517,2019-12,35
31518,2019-12,35
31519,2019-12,35
31520,2019-12,35


In [21]:
# transformacion comun de datos para todos los modelos:
# remuevo periodo_min_producto, periodo_max_producto, periodo_min_customer, periodo_max_customer
df = df.drop(columns=["periodo_min_producto", "periodo_max_producto",
                   "periodo_min_customer", "periodo_max_customer"], errors='ignore')

# transformo columnas object a categorical
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype("category")

# transformo plan precios cuidados a categorical
#df["plan_precios_cuidados"] = df["plan_precios_cuidados"].astype("category")

In [22]:

# dropeo columns donde tenga mas sea todo nan hasta date_id 28
print(f"Df shape before dropping columns: {df.shape}")
subset = df[df["date_id"] <= 28]
cols_to_drop = subset.columns[subset.isna().all()]
df = df.drop(columns=cols_to_drop, errors='ignore')
print(f"Df shape after dropping columns: {df.shape}")

Df shape before dropping columns: (31522, 312)
Df shape after dropping columns: (31522, 312)


In [23]:

from sklearn.model_selection import BaseCrossValidator
import numpy as np

class CustomTimeSeriesSplit(BaseCrossValidator):
    def __init__(self, n_splits=3, gap=1):
        self.n_splits = n_splits
        self.gap = gap

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        # Asegurar que X es DataFrame
        
        unique_dates = sorted(X["date_id"].unique(), reverse=True)
        
        for i in range(self.n_splits):
                
            test_date_id = unique_dates[i]
            train_date_id = test_date_id - self.gap - 1
            
            # Usar np.where para obtener posiciones enteras
            train_mask = X["date_id"] <= train_date_id
            test_mask = X["date_id"] == test_date_id
            
            train_idx = np.where(train_mask)[0]
            test_idx = np.where(test_mask)[0]
            
            yield train_idx, test_idx

In [24]:

class LinearRegressionModel:
    def __init__(self):
        self.model = None
    
    @property
    def name(self):
        return "LinearRegression"
    
    def prepare_dataset(self, df):
        df = df.copy()
        df = df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum"}).sort_values(["product_id", "date_id"])
        for lag in range(1, 12):
            df[f"tn_{lag}"] = df.groupby("product_id")["tn"].shift(lag)
        return df
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):
        from sklearn.linear_model import LinearRegression
        # si quiero  201912, entreno con 201812 (por estacionalidad)
        # lo busco dinamicamente con pred_df
        date_id_pred = pred_df["date_id"].unique()[0]
        train_df = train_df[train_df["date_id"] == (date_id_pred-12)]
        prod_ids_magicos = [20002, 20003, 20006, 20010, 20011, 20018, 20019, 20021,
            20026, 20028, 20035, 20039, 20042, 20044, 20045, 20046, 20049,
            20051, 20052, 20053, 20055, 20008, 20001, 20017, 20086, 20180,
            20193, 20320, 20532, 20612, 20637, 20807, 20838]
        train_df = train_df[train_df["product_id"].isin(prod_ids_magicos)]

        # elimino registros incompletos
        features = ["tn"] + [f"tn_{lag}" for lag in range(1, 12)]
        target = "target"
        train_df = train_df.dropna(subset=features + [target])
        print(f"Registros de entrenamiento: {len(train_df)}")
        X = train_df[features]
        y = train_df[target]
        model = LinearRegression()
        model.fit(X, y)
        self.model = model

        # hago la prediccion
        pred_df = pred_df.copy()
        
        # solo hace la prediccion para los productos que tienen todas las features
        pred_df = pred_df.dropna(subset=features)
        X_pred = pred_df[features]
        pred_df["prediction"] = model.predict(X_pred).clip(min=0)  # Aseguro que la prediccion no sea negativa
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })
    

In [25]:
class SimpleMovingAveragePredictor:
    ''' Usa una media movil simple para predecir tn'''
    def __init__(self, window_size=12):
        self.model = None
        self.window_size = window_size

    @property
    def name(self):
        return f"SMA-{self.window_size}"
    
    def prepare_dataset(self, df):
        df = df.copy()
        df = df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum"}).sort_values(["product_id", "date_id"])
        # hago una columna que es la media movil simple agrupada por producto
        df["tn_sma"] = df.groupby("product_id")["tn"].transform(
            lambda x: x.rolling(window=self.window_size, min_periods=1).mean()
        )
        return df
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):

        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["tn_sma"],
        })

In [26]:
class ExponentialMovingAveragePredictor:
    ''' Usa una media movil exponencial para predecir tn'''
    def __init__(self, window_size=12):
        self.model = None
        self.window_size = window_size

    @property
    def name(self):
        return f"EMA-{self.window_size}"
    def prepare_dataset(self, df):
        df = df.copy()
        df = df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum"}).sort_values(["product_id", "date_id"])
        # hago una columna que es la media movil exponencial agrupada por producto
        df["tn_ema"] = df.groupby("product_id")["tn"].transform(
            lambda x: x.ewm(span=self.window_size, adjust=False).mean()
        )
        return df
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["tn_ema"],
        })

In [27]:
class AutoGluonPredictor:
    def __init__(self, presets="best_quality", estimator=None):
        self.model = None
        self.presets = presets
        self.estimator = estimator

    @property
    def name(self):
        if self.estimator:
            return f"AutoGluon-{self.estimator}"
        return f"AutoGluon-{self.presets}"
    
    def prepare_dataset(self, df):

        df = df.copy()
        df["fecha"] = df["fecha"].apply(lambda x: x.to_timestamp("M"))
        df = df.rename(columns={"fecha": "timestamp"})
        df["product_id"] = df["product_id"].astype(int)
        df["serie_id"] = df["product_id"].astype(str) + "-" + df["customer_id"].astype(str)
        df["cat1"] = df["cat1"].astype("category")
        df["cat2"] = df["cat2"].astype("category")
        df["cat3"] = df["cat3"].astype("category")
        df["brand"] = df["brand"].astype("category")
        df["sku_size"] = df["sku_size"].astype("category")

        self.static_features_df = pd.DataFrame({
            "cat1": df.groupby("serie_id")["cat1"].first(),
            "cat2": df.groupby("serie_id")["cat2"].first(),
            "cat3": df.groupby("serie_id")["cat3"].first(),
            "brand": df.groupby("serie_id")["brand"].first(),
            "sku_size": df.groupby("serie_id")["sku_size"].first(),
            "customer_id": df.groupby("serie_id")["customer_id"].first(),
            "product_id": df.groupby("serie_id")["product_id"].first(),
        }).reset_index()
        

        min_periods = 12  # Mínimo 6 meses de datos
        product_counts = df.groupby(["product_id"]).size()
        valid_products = product_counts[product_counts >= min_periods].index
        df = df[df["product_id"].isin(valid_products)]        
        df = df.dropna(subset=["tn"])
        return df
    
    def fit_and_predict(self, train_df, pred_df, df_model):
        from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame
        # el autogluon lo entreno con todas las fechas hasta pred_df
        train_df = df_model[df_model["date_id"] <= pred_df["date_id"].unique()[0]]
        train_df = train_df[train_df["product_id"].isin(product_ids)]
        ts_data = TimeSeriesDataFrame.from_data_frame(
            train_df.drop(columns=["target"]), 
            id_column="serie_id", 
            timestamp_column="timestamp", 
            static_features_df=self.static_features_df
        )
        ts_data = ts_data.sort_index()
        ts_data = ts_data.fill_missing_values()

        predictor = TimeSeriesPredictor(
            prediction_length=2,
            target="tn",
            freq="MS",
        )
        if self.estimator:
            predictor.fit(ts_data, hyperparameters={self.estimator: {}})
        else:
            predictor.fit(ts_data, presets=self.presets)
        forecast = predictor.predict(ts_data)
        forecast_mean = forecast["mean"].reset_index()
        forecast_mean = forecast_mean[forecast_mean["timestamp"] == forecast_mean["timestamp"].max()]
        forecast_mean[["product_id", "customer_id"]] = forecast_mean["item_id"].str.split("-", expand=True)
        forecast_mean["product_id"] = forecast_mean["product_id"].astype(int)
        forecast_mean = forecast_mean.groupby("product_id").agg({
            "mean": "sum",
        }).reset_index()

        # rename item_id to product_id
        pred_df = pred_df.copy()
        pred_df["product_id"] = pred_df["product_id"].astype(int)
        # separo serie_id en product_id y customer_id
        pred_df = pred_df.groupby(["product_id"]).agg({
            "target": "sum",
            "date_id": "first"
        }).reset_index()
        pred_df = pred_df.merge(forecast_mean, on=["product_id"], how="left")
        pred_df = pred_df.rename(columns={"mean": "prediction"})
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })



In [28]:


class BaseTabularPredictor:
    
    def _scaling_df(self, df, train=True):
        df = df.copy()
        import re
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        transformations = {
            "tn": [
                r"tn$",
                r"cust_request_qty_per_tn$",
                r"tn_lag_*",
                r"tn_rolling_mean_*",
                r"tn_rolling_max_*",
                r"tn_rolling_min_*",
                r"tn_.*_vendidas$",
                r"tn_agg*",
                r"tn_wavelet_*",
            ]
            + [r"stock_final$"]
            + [r"cust_request_tn_minus_tn$"]
            + [r"tn_diff_*"],
            "cust_request_qty": [
                r"cust_request_qty$",
                r"cust_request_qty_lag_*",
                r"cust_request_qty_rolling_mean_*",
                r"cust_request_qty_rolling_max_*",
                r"cust_request_qty_rolling_min_*",
                r"cust_request_qty_.*_vendidas$",
                r"cust_request_qty_agg*",
                r"cust_request_qty_wavelet_*",
            ]
            + [r"cust_request_qty_diff_*"],
        }

        # busco todas las columnas que empiezan con prod_ y agrego key y valor en transformation
        for col in numeric_cols:
            if col.startswith("prod_"):
                transformations[col] = [r"{}$".format(col)]

        from pandas.errors import PerformanceWarning
        import warnings
        warnings.simplefilter(action="ignore", category=PerformanceWarning)
        df = df.set_index(['serie_id', "date_id"])
        if train:
            prod_stats = df.groupby(["serie_id"])[
                list(transformations.keys())
            ].agg(["std"])
            prod_stats.columns = [
                f"{col[0]}_{col[1]}" for col in prod_stats.columns
            ]  # renombro las columnas para que no tengan tupla

            prod_stats = prod_stats.reset_index()
            self.prod_stats = prod_stats
            prod_stats = prod_stats.set_index(['serie_id'])
            # supress performance warnings
            self.prod_stats = prod_stats

            print("Scaling")
        else:
            if self.prod_stats is None:
                raise ValueError("prod_stats is not set. Call prepare_dataset first.")
            prod_stats = self.prod_stats
        for trainer, regex_cols in transformations.items():
            for col in regex_cols:
                matching_cols = [c for c in numeric_cols if re.match(col, c)]
                if not matching_cols:
                    continue
                for col in matching_cols:
                    std_col = prod_stats[trainer + "_std"]
                    df[f"{col}_scaled"] = (df[col] / std_col).replace([np.inf, -np.inf], np.nan)

        # scalo el target con tn_std
        df["target_scaled"] = df["target"] / prod_stats["tn_std"]
        df["target_scaled"] = df["target_scaled"].replace([np.inf, -np.inf], np.nan).fillna(0)

        df = df.reset_index()
        return df

    def prepare_dataset(self, df):
        df = df.copy()
        df["serie_id"] = df["product_id"].astype(str) + "-" + df["customer_id"].astype(str)
        return df


class AutoMLPredictor(BaseTabularPredictor):
    
    def __init__(self, estimator="lgbm", time_budget=60):
        self.model = None
        self.prod_stats = None
        self.estimator = estimator
        self.time_budget = time_budget

    @property
    def name(self):
        return f"AutoML-{self.estimator}-{self.time_budget}s"
    
    def custom_metric(self, X_val, y_val, estimator, labels, X_train, y_train, *args, **kwargs):
        y_pred = estimator.predict(X_val)
    
        temp_df = pd.DataFrame({
            "product_id": X_val["product_id"].values,
            "customer_id": X_val["customer_id"].values,
            "y_true": y_val,
            "y_pred": y_pred
        })
        temp_df["product_id"] = temp_df["product_id"].astype(int)
        temp_df["customer_id"] = temp_df["customer_id"].astype(int)
        prod_stats = self.prod_stats.copy().reset_index()
        prod_stats[["product_id", "customer_id"]] = prod_stats["serie_id"].str.split("-", expand=True)
        prod_stats["product_id"] = prod_stats["product_id"].astype(int)
        prod_stats["customer_id"] = prod_stats["customer_id"].astype(int)
        temp_df = temp_df.merge(prod_stats[["product_id", "customer_id", "tn_std"]], on=["product_id", "customer_id"], how="left")
        # desescale the predictions
        temp_df["y_pred"] = temp_df["y_pred"] * temp_df["tn_std"]
        temp_df["y_true"] = temp_df["y_true"] * temp_df["tn_std"]
    
        grouped = temp_df.groupby("product_id")[["y_true", "y_pred"]].sum()
        total_true = grouped["y_true"].sum()
    
        if total_true == 0:
            return 0.0, {"total_error": 0.0}
    
        total_error = np.abs(grouped["y_pred"] - grouped["y_true"]).sum() / total_true
        return total_error, {"total_error": total_error}

    def fit_and_predict(self, train_df, pred_df, df_model):
        from flaml import AutoML
        train_df = self._scaling_df(train_df, train=True)
        pred_df = self._scaling_df(pred_df, train=False)
        tscv = CustomTimeSeriesSplit(2, gap=1)
        automl_settings = {
            "time_budget": self.time_budget,
            "task": "regression",
            "metric": self.custom_metric,
            "estimator_list": [self.estimator],
            "n_jobs": -1,
            "eval_method": "cv",
            "split_type": tscv,
            "verbose": 3,
            "retrain_full": True
        }
        X_train = train_df.drop(columns=["target", "target_scaled", "fecha"])
        y_train = train_df["target_scaled"]
        automl = AutoML()
        automl.fit(X_train, y_train, **automl_settings)
        # hago la prediccion
        y_pred = automl.predict(pred_df.drop(columns=["target", "target_scaled", "fecha"]))
        pred_df = pred_df.copy()
        pred_df["prediction"] = y_pred
        pred_df["serie_id"] = pred_df["product_id"].astype(str) + "-" + pred_df["customer_id"].astype(str)
        pred_df.set_index("serie_id", inplace=True)
        pred_df["prediction"] = pred_df["prediction"] * self.prod_stats["tn_std"]
        pred_df = pred_df.reset_index().groupby("product_id").agg({
            "target": "sum",
            "date_id": "first",
            "prediction": "sum"
        }).reset_index()
 
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })
        


In [29]:
class AutoGluonTabularPredictor(BaseTabularPredictor):
    
    def __init__(self, presets="medium_quality", exclude_model_types=None, time_budget=None, subsample=1):
        self.model = None
        self.prod_stats = None
        self.presets = presets
        self.exclude_model_types = exclude_model_types or []
        self.time_budget = time_budget
        self.subsample = subsample

    @property
    def name(self):
        return f"AutoGluonTabular-{self.presets}-budget-{self.time_budget}s-subsample-{self.subsample}"
    
    def fit_and_predict(self, train_df, pred_df, df_model):
        from autogluon.tabular import TabularPredictor, TabularDataset
        train_df = self._scaling_df(train_df, train=True)
        pred_df = self._scaling_df(pred_df, train=False)

        # uso el date_id mas alto de train_df como tunning_data
        train_df = train_df.sample(frac=self.subsample, random_state=42)
        train = TabularDataset(train_df.drop(columns=["fecha", "serie_id"], errors='ignore'))
        pred = TabularDataset(pred_df.drop(columns=["fecha", "serie_id"], errors='ignore'))

        predictor = TabularPredictor(
            label="target_scaled",
            eval_metric="mean_absolute_error",
        )
        predictor.fit(
            train.drop(columns=["target"]),
            presets=self.presets,
            excluded_model_types=["RF", "XT"] + self.exclude_model_types,
            time_limit=self.time_budget,
        )

        y_pred = predictor.predict(pred.drop(columns=["target", "target_scaled"]))
        pred_df = pred_df.copy()
        pred_df["prediction"] = y_pred
        pred_df["serie_id"] = pred_df["product_id"].astype(str) + "-" + pred_df["customer_id"].astype(str)
        pred_df.set_index("serie_id", inplace=True)
        pred_df["prediction"] = pred_df["prediction"] * self.prod_stats["tn_std"]
        pred_df = pred_df.reset_index().groupby("product_id").agg({
            "target": "sum",
            "date_id": "first",
            "prediction": "sum"
        }).reset_index()
 
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })

In [30]:
class BasicXGBoostPredictor(BaseTabularPredictor):
    
    def __init__(self):
        self.model = None
        self.prod_stats = None

    @property
    def name(self):
        return f"BasicXGBoost"
    
    def fit_and_predict(self, train_df, pred_df, df_model):
        import xgboost as xgb
        train_df = self._scaling_df(train_df, train=True)
        pred_df = self._scaling_df(pred_df, train=False)

        X_train = train_df.drop(columns=["target", "target_scaled", "fecha", "serie_id"])
        y_train = train_df["target_scaled"]

        X_test = pred_df.drop(columns=["target", "target_scaled", "fecha", "serie_id"])
        y_test = pred_df["target_scaled"]
        
        # I use the tn_std as weight
        train_df = train_df.merge(self.prod_stats.reset_index()[["serie_id", "tn_std"]], on="serie_id", how="left")
        w_train = train_df["tn_std"].fillna(0)
        dtrain = xgb.DMatrix(X_train, label=y_train, weight=w_train, enable_categorical=True)
        dtest = xgb.DMatrix(X_test, label=y_test, enable_categorical=True)
        
        model = xgb.train(
            params={
                "objective": "reg:tweedie",
                "device": "cuda",
                "tree_method": "hist",
                "sampling_method": "uniform",
                "max_depth": 0,
                "learning_rate": 0.03,
                "num_leaves": 31,
                "subsample": 0.8,
                "colsample_bytree": 0.6,
            },
            dtrain=dtrain,
            evals=[(dtest, "test")],
            num_boost_round=1000,
            #num_boost_round=20
        )

        y_pred = model.predict(dtest)
        pred_df = pred_df.copy()
        pred_df["prediction"] = y_pred
        pred_df["serie_id"] = pred_df["product_id"].astype(str) + "-" + pred_df["customer_id"].astype(str)
        pred_df.set_index("serie_id", inplace=True)
        pred_df["prediction"] = pred_df["prediction"] * self.prod_stats["tn_std"]
        pred_df = pred_df.reset_index().groupby("product_id").agg({
            "target": "sum",
            "date_id": "first",
            "prediction": "sum"
        }).reset_index()
 
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })

In [31]:
class BasicLGBMPredictor(BaseTabularPredictor):
    def __init__(self):
        self.model = None
        self.prod_stats = None

    @property
    def name(self):
        return f"BasicLGBM"
    
    def fit_and_predict(self, train_df, pred_df, df_model):
        import lightgbm as lgb
        train_df = self._scaling_df(train_df, train=True)
        pred_df = self._scaling_df(pred_df, train=False)

        X_train = train_df.drop(columns=["target", "target_scaled", "fecha", "serie_id"])
        y_train = train_df["target_scaled"]

        X_test = pred_df.drop(columns=["target", "target_scaled", "fecha", "serie_id"])
        y_test = pred_df["target_scaled"]
        
        # I use the tn_std as weight
        train_df = train_df.merge(self.prod_stats.reset_index()[["serie_id", "tn_std"]], on="serie_id", how="left")
        w_train = train_df["tn_std"].fillna(0)
        
        dtrain = lgb.Dataset(X_train, label=y_train, weight=w_train)
        dtest = lgb.Dataset(X_test, label=y_test)

        params = {
            "objective": "tweedie",
            "device": "cpu",
            "num_leaves": 31,
            "learning_rate": 0.03,
            "feature_fraction": 0.6,
            "bagging_fraction": 0.8,
            "bagging_freq": 5,
            "verbosity": -1,
        }

        model = lgb.train(
            params,
            dtrain,
            #num_boost_round=1000,
            num_boost_round=4000,
            valid_sets=[dtest],
            callbacks=[lgb.log_evaluation(1000)]
        )

        y_pred = model.predict(X_test)
        pred_df = pred_df.copy()
        pred_df["prediction"] = y_pred
        pred_df["serie_id"] = pred_df["product_id"].astype(str) + "-" + pred_df["customer_id"].astype(str)
        pred_df.set_index("serie_id", inplace=True)
        pred_df["prediction"] = pred_df["prediction"] * self.prod_stats["tn_std"]
        pred_df = pred_df.reset_index().groupby("product_id").agg({
            "target": "sum",
            "date_id": "first",
            "prediction": "sum"
        }).reset_index()

        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })

In [32]:
#autogluon_tabular_predictor = AutoGluonTabularPredictor(presets="medium")
#df_autogluon_tabular = autogluon_tabular_predictor.prepare_dataset(df)
#test_df = df_autogluon_tabular[df_autogluon_tabular["date_id"] == 33]
#train_df = df_autogluon_tabular[df_autogluon_tabular["date_id"] < 32]
#results = autogluon_tabular_predictor.fit_and_predict(train_df, test_df, df_autogluon_tabular)

In [33]:

# import deep copy
from copy import deepcopy
class EnsambleTrainer:
    def __init__(self, models):
        self.models = models
        self.model_weights = None
        self.train_results = None

    def _combina_results(self, results):
        from collections import defaultdict
        # Diccionario para almacenar resultados intermedios
        combined_results = defaultdict(dict)

        for split, models in results.items():
            for model_info in models:
                model_name = model_info["model"].name
                pred_df = model_info["pred_df"]

                for _, row in pred_df.iterrows():
                    key = (row["product_id"], row["date_id"])
                    combined_results[key]["product_id"] = row["product_id"]
                    combined_results[key]["date_id"] = row["date_id"]
                    combined_results[key]["target"] = row["target"]
                    combined_results[key][f"prediction_{model_name}"] = row["prediction"]

        # Convertir a DataFrame
        results_df = pd.DataFrame(combined_results.values())

        # Opcional: ordenar columnas
        cols = ["product_id", "date_id", "target"] + sorted([col for col in results_df.columns if col not in {"product_id", "date_id", "target"}])
        results_df = results_df[cols]

        def fill_row_na_with_row_mean(row, prediction_cols):
            preds = row[prediction_cols]
            row[prediction_cols] = preds.fillna(preds.mean(skipna=True))
            return row

        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]

        results_df = results_df.apply(fill_row_na_with_row_mean, axis=1, prediction_cols=prediction_cols)   
        self.train_results = results_df
        return results_df
        
    def _optimize_weights(self, results_df):
        # Asegurar columnas de predicción con prefijo prediction_
        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]

        # Lista donde irán los dicts de pesos y predicciones agregadas
        weights_list = []
        predictions_list = []

        # Para guardar pesos temporales por product_id
        product_weights = {}

        # Por cada fila (product_id, date_id), elegimos el mejor modelo (peso 1 para ese, 0 para el resto)
        for _, row in results_df.iterrows():
            product_id = row["product_id"]
            target = row["target"]

            preds = np.array([row[col] for col in prediction_cols])
            errors = (preds - target) ** 2
            best_idx = np.argmin(errors)

            one_hot_weights = np.zeros(len(prediction_cols))
            one_hot_weights[best_idx] = 1

            # Acumulamos
            if product_id not in product_weights:
                product_weights[product_id] = []
            product_weights[product_id].append(one_hot_weights)

        # Promediamos los pesos por product_id
        for product_id, weight_list in product_weights.items():
            avg_weights = np.mean(weight_list, axis=0)
            weights_dict = dict(zip(prediction_cols, avg_weights))

            # Calculamos la predicción promedio ponderada usando los pesos promedio
            product_rows = results_df[results_df["product_id"] == product_id]
            preds_matrix = product_rows[prediction_cols].values
            weighted_preds = preds_matrix @ avg_weights
            mean_prediction = np.mean(weighted_preds)

            weights_list.append(weights_dict)
            predictions_list.append(mean_prediction)

        # Creamos el DataFrame final
        product_ids = list(product_weights.keys())
        agg_df = pd.DataFrame({
            "product_id": product_ids,
            "weights": weights_list,
            "predictions": predictions_list
        })

        self.model_weights = agg_df[["product_id", "weights"]].set_index("product_id")


    def _compute_metrics_simple(self, y_true, y_pred):
        """Calcula el error absoluto medio entre y_true e y_pred"""
        return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) if np.sum(y_true) > 0 else 0

    def _compute_metrics(self, results_df):
        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]
        agg_df = results_df.groupby("product_id")[["target"] + prediction_cols].sum().reset_index()
        agg_df = agg_df.set_index("product_id")
        agg_df["weights"] = self.model_weights["weights"]
        agg_df["prediction_ensamble"] = agg_df.apply(
            lambda row: sum(row[col] * row["weights"][col] for col in prediction_cols), 
            axis=1
        )
        self.agg_df = agg_df
        # calculo metricas
        metrics = {}
        def total_error(y_true, y_pred):
            return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

        prediction_cols = [col for col in agg_df.columns if col.startswith("prediction_")]
        for col in prediction_cols:
            metrics[col] = total_error(agg_df["target"], agg_df[col])
        return pd.DataFrame(metrics, index=[0]).T.rename(columns={0: "error"})
    
    def fit(self, df, splitter):
        """Entrena todos los modelos en cada split del splitter"""
        df = df.dropna(subset=["target"])
        number_of_splits = splitter.get_n_splits(df)
        results = {f"split_{i}": [] for i in range(number_of_splits)}
        for i, (train_idx, test_idx) in enumerate(splitter.split(df)):
            # Obtener las fechas de los splits originales
            train_dates = df.iloc[train_idx]["date_id"].unique()
            test_dates = df.iloc[test_idx]["date_id"].unique()
            
            for m in self.models:
                model = deepcopy(m)
                df_model = model.prepare_dataset(df)
                
                # Recalcular train/test usando las fechas, no los índices
                train_df = df_model[df_model["date_id"].isin(train_dates)]
                test_df = df_model[df_model["date_id"].isin(test_dates)]
                test_df = test_df[test_df["product_id"].isin(product_ids)]
                
                pred_df = model.fit_and_predict(train_df, test_df, df_model)
                results[f"split_{i}"].append({
                    "model": model,
                    "target": test_df[["target", "product_id", "date_id"]],
                    "pred_df": pred_df
                })
                print(f"Modelo {model.name} entrenado en split {i+1}/{number_of_splits}:")
                print(self._compute_metrics_simple(pred_df["target"], pred_df["prediction"]))
        results_df = self._combina_results(results)
        self._optimize_weights(results_df)
        print(self._compute_metrics(results_df))

    def final_pred(self, df, kaggle_date_id):
        """Vuelve a entrenar todos los modelos con el dataset completo"""
        results = {"split_final": []}
        for m in self.models:
            model = deepcopy(m)
            df_model = model.prepare_dataset(df)
            # Recalcular train/test usando las fechas, no los índices
            train_df = df_model[df_model["date_id"] < kaggle_date_id]
            train_df = train_df.dropna(subset=["target"])
            pred_df = df_model[df_model["date_id"] == kaggle_date_id]
            pred_df = pred_df[pred_df["product_id"].isin(product_ids)]

            pred_df = model.fit_and_predict(train_df, pred_df, df_model)
            results["split_final"].append({
                "model": model,
                "target": pred_df[["target", "product_id", "date_id"]],
                "pred_df": pred_df
            })
        results_df = self._combina_results(results)  
        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]
        agg_df = results_df.groupby("product_id")[["target"] + prediction_cols].sum().reset_index()
        agg_df = agg_df.set_index("product_id", drop=False)
        agg_df["weights"] = self.model_weights["weights"]
        agg_df["prediction_ensamble"] = agg_df.apply(
            lambda row: sum(row[col] * row["weights"][col] for col in prediction_cols), 
            axis=1
        )
        return agg_df.rename(columns={"prediction_ensamble": "tn"})


In [34]:
# TODO: entrenar el lightgbm con todos los product_ids? (la validacion solo con los 780)
import sys
from contextlib import redirect_stdout

trainer = EnsambleTrainer([
    AutoGluonTabularPredictor(exclude_model_types=["KNN"]),
    BasicLGBMPredictor(),
    #BasicXGBoostPredictor(),
    LinearRegressionModel(),
    #AutoGluonTabularPredictor(presets="best", time_budget=3600, exclude_model_types=["KNN"]),
    AutoGluonPredictor(presets="best_quality"),
    #AutoGluonPredictor(presets="fast_training"),
    SimpleMovingAveragePredictor(window_size=12)
])
splitter = CustomTimeSeriesSplit(n_splits=3, gap=1)
trainer.fit(df, splitter)


Scaling


No path specified. Models will be saved in: "AutogluonModels/ag-20250717_131410"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jun 26 14:16:59 UTC 2
CPU Count:          16
Memory Avail:       17.86 GB / 31.23 GB (57.2%)
Disk Space Avail:   734.63 GB / 914.78 GB (80.3%)
Presets specified: ['medium_quality']
	Consider setting `time_limit` to ensure training finishes within an expected duration or experiment with a small portion of `train_data` to identify an ideal `presets` and `hyperparameters` configuration.
Beginning AutoGluon training ...
AutoGluon will save models to "/home/fede/programacion/labo3/AutogluonModels/ag-20250717_131410"
Train Data Rows:    27249
Train Data Columns: 577
Label Column:       target_scaled
AutoGluon infers your prediction problem is: 'regression' (becaus

[1000]	valid_set's l1: 0.63003
[2000]	valid_set's l1: 0.620432
[3000]	valid_set's l1: 0.616021
[4000]	valid_set's l1: 0.61251
[5000]	valid_set's l1: 0.610678
[6000]	valid_set's l1: 0.609446
[7000]	valid_set's l1: 0.608931
[8000]	valid_set's l1: 0.608336
[9000]	valid_set's l1: 0.608112
[10000]	valid_set's l1: 0.608049


	-0.6079	 = Validation score   (-mean_absolute_error)
	107.21s	 = Training   runtime
	0.3s	 = Validation runtime
Fitting model: LightGBM ...


[1000]	valid_set's l1: 0.60331
[2000]	valid_set's l1: 0.597774
[3000]	valid_set's l1: 0.594104
[4000]	valid_set's l1: 0.592814
[5000]	valid_set's l1: 0.591904
[6000]	valid_set's l1: 0.591968


...

Modelo AutoGluon-best_quality entrenado en split 2/3:
0.23305520755531847
Modelo SMA-12 entrenado en split 2/3:
0.2579996583872914
Scaling


No path specified. Models will be saved in: "AutogluonModels/ag-20250717_141005"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jun 26 14:16:59 UTC 2
CPU Count:          16
Memory Avail:       14.46 GB / 31.23 GB (46.3%)
Disk Space Avail:   733.41 GB / 914.78 GB (80.2%)
Presets specified: ['medium_quality']
	Consider setting `time_limit` to ensure training finishes within an expected duration or experiment with a small portion of `train_data` to identify an ideal `presets` and `hyperparameters` configuration.
Beginning AutoGluon training ...
AutoGluon will save models to "/home/fede/programacion/labo3/AutogluonModels/ag-20250717_141005"
Train Data Rows:    25433
Train Data Columns: 577
Label Column:       target_scaled
AutoGluon infers your prediction problem is: 'regression' (becaus

Modelo AutoGluonTabular-medium_quality-budget-Nones-subsample-1 entrenado en split 3/3:
0.3162591
Scaling
[1000]	valid_0's tweedie: 9.32966
[2000]	valid_0's tweedie: 8.6851
[3000]	valid_0's tweedie: 8.22333
[4000]	valid_0's tweedie: 7.80413
Modelo BasicLGBM entrenado en split 3/3:
0.3730154346914153
Registros de entrenamiento: 33
Modelo LinearRegression entrenado en split 3/3:
0.43769825


No path specified. Models will be saved in: "AutogluonModels/ag-20250717_142325"
Beginning AutoGluon training...
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250717_142325'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jun 26 14:16:59 UTC 2
CPU Count:          16
GPU Count:          1
Memory Avail:       15.55 GB / 31.23 GB (49.8%)
Disk Space Avail:   733.31 GB / 914.78 GB (80.2%)
Setting presets to: best_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'MS',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 't

Modelo AutoGluon-best_quality entrenado en split 3/3:
0.32437684220687396
Modelo SMA-12 entrenado en split 3/3:
0.251847995358391
                                                                        error
prediction_AutoGluon-best_quality                                    0.165689
prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1  0.169186
prediction_BasicLGBM                                                 0.221719
prediction_LinearRegression                                          0.244367
prediction_SMA-12                                                    0.175209
prediction_ensamble                                                  0.121272


In [35]:
trainer.model_weights

,weights
product_id,
20001.0,"{'prediction_AutoGluon-best_quality': 0.0, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.3333333333333333, 'prediction_SMA-12': 0.3333333333333333}"
20002.0,"{'prediction_AutoGluon-best_quality': 0.0, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.6666666666666666, 'prediction_SMA-12': 0.0}"
20003.0,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}"
20004.0,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.0, 'prediction_LinearRegression': 0.6666666666666666, 'prediction_SMA-12': 0.0}"
20005.0,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.3333333333333333, 'prediction_BasicLGBM': 0.0, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}"
...,...
21263.0,"{'prediction_AutoGluon-best_quality': 0.0, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 1.0, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.0}"
21265.0,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}"
21266.0,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}"


In [36]:
trainer._compute_metrics(trainer.train_results)

,error
prediction_AutoGluon-best_quality,0.165689
prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1,0.169186
prediction_BasicLGBM,0.221719
prediction_LinearRegression,0.244367
prediction_SMA-12,0.175209
prediction_ensamble,0.121272


In [37]:
trainer.train_results[trainer.train_results["product_id"] == 20001.0].head(10)

,product_id,date_id,target,prediction_AutoGluon-best_quality,prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1,prediction_BasicLGBM,prediction_LinearRegression,prediction_SMA-12
0,20001.0,33.0,1504.688599,1521.775607,1569.224976,1505.419057,1195.374512,1487.869476
780,20001.0,32.0,1397.372314,1355.918428,1311.520020,1420.770300,1383.669800,1549.010539
1560,20001.0,31.0,1561.505493,1359.474096,1225.111938,1425.364693,2354.497559,1530.566284


In [38]:

models_used = [model.name for model in trainer.models]
models_used = " ".join(models_used)
# hago un hash en base de models_used para el nombre del archivo
import hashlib
hash_object = hashlib.md5(models_used.encode())
hash_hex = hash_object.hexdigest()

trainer.train_results.to_csv(f"train_results_{hash_hex}.csv", index=False)

In [39]:
trainer.agg_df

,target,prediction_AutoGluon-best_quality,prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1,prediction_BasicLGBM,prediction_LinearRegression,prediction_SMA-12,weights,prediction_ensamble
product_id,,,,,,,,
20001.0,4463.566406,4237.168130,4105.856934,4351.554049,4933.541870,4567.446299,"{'prediction_AutoGluon-best_quality': 0.0, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.3333333333333333, 'prediction_SMA-12': 0.3333333333333333}",4617.514073
20002.0,4490.422363,3132.265147,3115.646240,3000.663391,4550.385742,3481.152552,"{'prediction_AutoGluon-best_quality': 0.0, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.6666666666666666, 'prediction_SMA-12': 0.0}",4033.811625
20003.0,2922.161682,2759.516838,2292.572998,2338.129779,3459.910645,2422.960470,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",2506.869029
20004.0,2426.538391,1983.388992,1742.217712,1686.258320,2037.894104,1859.216726,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.0, 'prediction_LinearRegression': 0.6666666666666666, 'prediction_SMA-12': 0.0}",2019.725733
20005.0,2196.938965,1838.107760,1656.841064,1520.725131,2059.518066,1888.259326,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.3333333333333333, 'prediction_BasicLGBM': 0.0, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",1794.402717
...,...,...,...,...,...,...,...,...
21263.0,0.060690,0.174600,0.138086,0.050429,0.073929,0.285505,"{'prediction_AutoGluon-best_quality': 0.0, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 1.0, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.0}",0.050429
21265.0,0.225280,0.189808,0.147262,0.120167,0.189808,0.301994,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",0.203990
21266.0,0.236650,0.204437,0.162165,0.132611,0.204437,0.318535,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",0.218527


In [40]:
agg_df = trainer.agg_df.copy()
pred_columns = [col for col in agg_df.columns if col.startswith("prediction_") if col != "prediction_ensamble"]

# entreno un linear regression para usar las predicciones y hacer una prediccion final
from sklearn.linear_model import LinearRegression
X = agg_df[pred_columns]
y = agg_df["target"]
model = LinearRegression()
model.fit(X, y)
agg_df["prediction_final"] = model.predict(X)
total_error = np.sum(np.abs(agg_df["target"] - agg_df["prediction_final"])) / np.sum(agg_df["target"])
print(f"Total error: {total_error}")

# calculo weights para cada product_id usando linear regression en lugar de scipy.minimize
from scipy.optimize import minimize

def optimize_weights_per_product(y_true, y_pred_values):
    """Optimiza los pesos para UNA SOLA fila usando scipy.minimize"""
    predictions = np.array(y_pred_values)
    
    if np.allclose(predictions, 0) or y_true == 0:
        # Si todas las predicciones son 0 o target es 0, usar pesos uniformes
        return np.ones(len(predictions)) / len(predictions)
    
    def objective(weights):
        """Función objetivo: error absoluto"""
        weighted_pred = np.dot(predictions, weights)
        return abs(y_true - weighted_pred)
    
    # Restricciones
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}  # Suma = 1
    bounds = [(0, 1) for _ in range(len(predictions))]  # Pesos entre 0 y 1
    initial_weights = np.ones(len(predictions)) / len(predictions)  # Pesos iniciales uniformes
    
    try:
        result = minimize(objective, initial_weights, method='SLSQP', 
                         bounds=bounds, constraints=constraints)
        if result.success:
            return result.x
        else:
            return initial_weights
    except:
        return initial_weights

# Aplicar la optimización
agg_df["weights_2"] = agg_df.apply(
    lambda row: optimize_weights_per_product(row["target"], row[pred_columns].values), 
    axis=1
)

# Calcular la predicción ponderada
agg_df["prediction_final_2"] = agg_df.apply(
    lambda row: np.dot(row[pred_columns].values, row["weights_2"]), 
    axis=1
)

total_error_2 = np.sum(np.abs(agg_df["target"] - agg_df["prediction_final_2"])) / np.sum(agg_df["target"])
print(f"Total error with scipy optimized weights: {total_error_2}")
agg_df

Total error: 0.14618739174667456
Total error with scipy optimized weights: 0.05301207495455856


,target,prediction_AutoGluon-best_quality,prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1,prediction_BasicLGBM,prediction_LinearRegression,prediction_SMA-12,weights,prediction_ensamble,prediction_final,weights_2,prediction_final_2
product_id,,,,,,,,,,,
20001.0,4463.566406,4237.168130,4105.856934,4351.554049,4933.541870,4567.446299,"{'prediction_AutoGluon-best_quality': 0.0, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.3333333333333333, 'prediction_SMA-12': 0.3333333333333333}",4617.514073,4665.187279,"[0.19010440191940628, 0.19012073649820432, 0.19009319381472561, 0.2395680396974106, 0.19011362807025323]",4463.566406
20002.0,4490.422363,3132.265147,3115.646240,3000.663391,4550.385742,3481.152552,"{'prediction_AutoGluon-best_quality': 0.0, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.6666666666666666, 'prediction_SMA-12': 0.0}",4033.811625,3832.000059,"[0.010966844452880596, 0.010942755091183367, 0.010978396822036574, 0.9561718905447596, 0.010940113089139914]",4490.422432
20003.0,2922.161682,2759.516838,2292.572998,2338.129779,3459.910645,2422.960470,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",2506.869029,3015.122341,"[0.1335285016870911, 0.13356213499349404, 0.1335444977246512, 0.4657936047276063, 0.13357126086715737]",2922.161596
20004.0,2426.538391,1983.388992,1742.217712,1686.258320,2037.894104,1859.216726,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.0, 'prediction_LinearRegression': 0.6666666666666666, 'prediction_SMA-12': 0.0}",2019.725733,2076.288442,"[0.0, 5.803484507206468e-10, 8.687056698883806e-10, 0.9999999991267251, 1.3634585253095413e-11]",2037.894105
20005.0,2196.938965,1838.107760,1656.841064,1520.725131,2059.518066,1888.259326,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.3333333333333333, 'prediction_BasicLGBM': 0.0, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",1794.402717,2010.738746,"[1.1252325815095895e-09, 0.0, 0.0, 1.0, 2.3582935029913436e-09]",2059.518073
...,...,...,...,...,...,...,...,...,...,...,...
21263.0,0.060690,0.174600,0.138086,0.050429,0.073929,0.285505,"{'prediction_AutoGluon-best_quality': 0.0, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 1.0, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.0}",0.050429,2.643872,"[6.747370544395434e-11, 0.0057756879005537035, 0.5791568834375302, 0.41506742845409456, 1.403478103679955e-10]",0.060690
21265.0,0.225280,0.189808,0.147262,0.120167,0.189808,0.301994,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",0.203990,2.676616,"[0.19248684021171228, 0.10982130149923813, 0.09092352287498151, 0.19248684054953483, 0.4142814948645333]",0.225280
21266.0,0.236650,0.204437,0.162165,0.132611,0.204437,0.318535,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",0.218527,2.692937,"[0.19424702553654716, 0.1221595269414946, 0.09927019092500346, 0.19424702279070385, 0.3900762338062509]",0.236649


In [41]:
from contextlib import redirect_stdout

with open("entrenamiento_final.log", "w") as f:
    with redirect_stdout(f):
        final_df = trainer.final_pred(df, 35)

No path specified. Models will be saved in: "AutogluonModels/ag-20250717_151733"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jun 26 14:16:59 UTC 2
CPU Count:          16
Memory Avail:       15.63 GB / 31.23 GB (50.0%)
Disk Space Avail:   732.89 GB / 914.78 GB (80.1%)
Presets specified: ['medium_quality']
	Consider setting `time_limit` to ensure training finishes within an expected duration or experiment with a small portion of `train_data` to identify an ideal `presets` and `hyperparameters` configuration.
Beginning AutoGluon training ...
AutoGluon will save models to "/home/fede/programacion/labo3/AutogluonModels/ag-20250717_151733"
Train Data Rows:    29076
Train Data Columns: 577
Label Column:       target_scaled
AutoGluon infers your prediction problem is: 'regression' (becaus

In [42]:
final_df

,product_id,target,prediction_AutoGluon-best_quality,prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1,prediction_BasicLGBM,prediction_LinearRegression,prediction_SMA-12,weights,tn
product_id,,,,,,,,,
20001.0,20001.0,0.0,1345.558345,1374.934204,1381.704589,1162.707520,1454.732737,"{'prediction_AutoGluon-best_quality': 0.0, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.3333333333333333, 'prediction_SMA-12': 0.3333333333333333}",1333.048282
20002.0,20002.0,0.0,1086.513454,1201.990723,1151.760312,1183.640625,1175.437134,"{'prediction_AutoGluon-best_quality': 0.0, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.6666666666666666, 'prediction_SMA-12': 0.0}",1173.013854
20003.0,20003.0,0.0,681.006864,842.291443,784.693287,684.763855,784.976405,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",750.225518
20004.0,20004.0,0.0,495.375528,646.343262,568.717459,580.485046,627.215322,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.0, 'prediction_LinearRegression': 0.6666666666666666, 'prediction_SMA-12': 0.0}",552.115207
20005.0,20005.0,0.0,471.320466,579.685181,553.474457,563.560852,668.270111,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.3333333333333333, 'prediction_BasicLGBM': 0.0, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",573.091919
...,...,...,...,...,...,...,...,...,...
21263.0,21263.0,0.0,0.001539,-0.005640,0.012962,0.467764,0.029993,"{'prediction_AutoGluon-best_quality': 0.0, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 1.0, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.0}",0.012962
21265.0,21265.0,0.0,0.073698,0.068000,0.063554,0.073698,0.089541,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",0.075598
21266.0,21266.0,0.0,0.073310,0.065490,0.059782,0.073310,0.094659,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",0.075917


In [43]:
final_df_2 = final_df.copy()
final_df_2["pred_linear"] = model.predict(final_df_2[pred_columns])
final_df_2["pred_linear"] = final_df_2["pred_linear"].clip(lower=0)  # Aseguro que la prediccion no sea negativa
final_df_2

final_df_3 = final_df_2.copy()
final_df_3["weights_2"] = agg_df["weights_2"]
final_df_3["pred_weights"] = final_df_3.apply(
    lambda row: np.dot(row[pred_columns].values, row["weights_2"]),
    axis=1
)
final_df_3["pred_weights"] = final_df_3["pred_weights"].clip(lower=0)  # Aseguro que la prediccion no sea negativa
final_df_3


,product_id,target,prediction_AutoGluon-best_quality,prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1,prediction_BasicLGBM,prediction_LinearRegression,prediction_SMA-12,weights,tn,pred_linear,weights_2,pred_weights
product_id,,,,,,,,,,,,
20001.0,20001.0,0.0,1345.558345,1374.934204,1381.704589,1162.707520,1454.732737,"{'prediction_AutoGluon-best_quality': 0.0, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.3333333333333333, 'prediction_SMA-12': 0.3333333333333333}",1333.048282,1395.324339,"[0.19010440191940628, 0.19012073649820432, 0.19009319381472561, 0.2395680396974106, 0.19011362807025323]",1334.964786
20002.0,20002.0,0.0,1086.513454,1201.990723,1151.760312,1183.640625,1175.437134,"{'prediction_AutoGluon-best_quality': 0.0, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.6666666666666666, 'prediction_SMA-12': 0.0}",1173.013854,1250.706313,"[0.010966844452880596, 0.010942755091183367, 0.010978396822036574, 0.9561718905447596, 0.010940113089139914]",1182.336505
20003.0,20003.0,0.0,681.006864,842.291443,784.693287,684.763855,784.976405,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",750.225518,803.536367,"[0.1335285016870911, 0.13356213499349404, 0.1335444977246512, 0.4657936047276063, 0.13357126086715737]",732.032453
20004.0,20004.0,0.0,495.375528,646.343262,568.717459,580.485046,627.215322,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.0, 'prediction_LinearRegression': 0.6666666666666666, 'prediction_SMA-12': 0.0}",552.115207,630.781255,"[0.0, 5.803484507206468e-10, 8.687056698883806e-10, 0.9999999991267251, 1.3634585253095413e-11]",580.485047
20005.0,20005.0,0.0,471.320466,579.685181,553.474457,563.560852,668.270111,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.3333333333333333, 'prediction_BasicLGBM': 0.0, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",573.091919,579.592900,"[1.1252325815095895e-09, 0.0, 0.0, 1.0, 2.3582935029913436e-09]",563.560854
...,...,...,...,...,...,...,...,...,...,...,...,...
21263.0,21263.0,0.0,0.001539,-0.005640,0.012962,0.467764,0.029993,"{'prediction_AutoGluon-best_quality': 0.0, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 1.0, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.0}",0.012962,2.627202,"[6.747370544395434e-11, 0.0057756879005537035, 0.5791568834375302, 0.41506742845409456, 1.403478103679955e-10]",0.201628
21265.0,21265.0,0.0,0.073698,0.068000,0.063554,0.073698,0.089541,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",0.075598,2.561438,"[0.19248684021171228, 0.10982130149923813, 0.09092352287498151, 0.19248684054953483, 0.4142814948645333]",0.078713
21266.0,21266.0,0.0,0.073310,0.065490,0.059782,0.073310,0.094659,"{'prediction_AutoGluon-best_quality': 0.3333333333333333, 'prediction_AutoGluonTabular-medium_quality-budget-Nones-subsample-1': 0.0, 'prediction_BasicLGBM': 0.3333333333333333, 'prediction_LinearRegression': 0.0, 'prediction_SMA-12': 0.3333333333333333}",0.075917,2.560471,"[0.19424702553654716, 0.1221595269414946, 0.09927019092500346, 0.19424702279070385, 0.3900762338062509]",0.079340


In [44]:
df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "tn_rolling_mean_12": "sum"})

,product_id,date_id,tn,tn_rolling_mean_12
0,20001,0,934.772217,0.0
1,20001,1,798.016174,0.0
2,20001,2,1303.357666,0.0
3,20001,3,1069.961304,0.0
4,20001,4,1502.201294,0.0
...,...,...,...,...
31517,21295,0,0.006990,0.0
31518,21296,7,0.006510,0.0
31519,21297,0,0.005790,0.0
31520,21298,7,0.005730,0.0


In [45]:

models_used = [model.name for model in trainer.models]
models_used = " ".join(models_used)
# hago un hash en base de models_used para el nombre del archivo
import hashlib
hash_object = hashlib.md5(models_used.encode())
hash_hex = hash_object.hexdigest()
description = f"Ensamble de modelos: {models_used}"
# save txt with name hash_hex.txt and the description
with open(f"description_{hash_hex}.txt", "w") as f:
    f.write(description)
submission = final_df[["product_id", "tn"]].reset_index(drop=True)
submission.to_csv(f"submission_weighted_ensamble_{hash_hex}.csv", index=False)
submission

,product_id,tn
0,20001.0,1333.048282
1,20002.0,1173.013854
2,20003.0,750.225518
3,20004.0,552.115207
4,20005.0,573.091919
...,...,...
775,21263.0,0.012962
776,21265.0,0.075598
777,21266.0,0.075917
778,21267.0,0.047521


In [46]:
submission_2 = final_df_2[["product_id", "pred_linear"]].rename(columns={"pred_linear": "tn"}).reset_index(drop=True)
submission_2.to_csv(f"submission_linear_regression_{hash_hex}.csv", index=False)
submission_2

,product_id,tn
0,20001.0,1395.324339
1,20002.0,1250.706313
2,20003.0,803.536367
3,20004.0,630.781255
4,20005.0,579.592900
...,...,...
775,21263.0,2.627202
776,21265.0,2.561438
777,21266.0,2.560471
778,21267.0,2.547241


In [47]:
print(description)

Ensamble de modelos: AutoGluonTabular-medium_quality-budget-Nones-subsample-1 BasicLGBM LinearRegression AutoGluon-best_quality SMA-12


In [48]:
submission_3 = final_df_3[["product_id", "pred_weights"]].rename(columns={"pred_weights": "tn"}).reset_index(drop=True)
submission_3.to_csv(f"submission_weights_ensamble_per_product_{hash_hex}.csv", index=False)
submission_3

,product_id,tn
0,20001.0,1334.964786
1,20002.0,1182.336505
2,20003.0,732.032453
3,20004.0,580.485047
4,20005.0,563.560854
...,...,...
775,21263.0,0.201628
776,21265.0,0.078713
777,21266.0,0.079340
778,21267.0,0.050538
